# Heart Disease Dataset — Exploratory Data Analysis

**Student:** Vishnu Gopisetti  | **ID:** 2025AE05666
**Course:** AIMLCZG523 — Machine Learning Operations (MLOps), Assignment 01

This notebook explores the processed Cleveland subset of the UCI Heart Disease
dataset (DOI 10.24432/C52P4X) used by the MLOps pipeline. It loads the cleaned
data through the project package (`src.heart_disease`), inspects distributions,
class balance, missing values, and correlations.

> Educational use only. This dataset is historical and small; nothing here is a
> medical conclusion.

Run this notebook from the repository root (or from `notebooks/`) with the
project virtualenv kernel: `.venv/bin/python -m notebook` after
`.venv/bin/pip install notebook`, or `make eda` for the script-based equivalent.


In [1]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

pandas 2.3.3 | numpy 2.4.6 | seaborn 0.13.2


In [2]:
import sys
from pathlib import Path

# Bootstrap sys.path so imports work whether Jupyter's working directory is the
# repository root or this notebooks/ directory.
PROJECT_ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src" / "heart_disease" / "config.py").is_file():
        PROJECT_ROOT = candidate
        break
assert PROJECT_ROOT is not None, (
    "Could not locate the project root (src/heart_disease) from "
    f"{Path.cwd()}. Launch Jupyter from the repository root."
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/krishnateja/Downloads/mlops-assignment-01/mlops-heart-disease


In [3]:
from src.heart_disease.config import FEATURES, TARGET, DATA_PROCESSED
from src.heart_disease.data import load_clean_dataframe

print(f"{len(FEATURES)} features: {FEATURES}")
print(f"Target column: {TARGET}")
print(f"Processed data path: {DATA_PROCESSED}")

df = load_clean_dataframe()
rows, cols = df.shape
print(f"Loaded cleaned dataset: {rows} rows x {cols} columns")

13 features: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Target column: target
Processed data path: /Users/krishnateja/Downloads/mlops-assignment-01/mlops-heart-disease/data/processed
Loaded cleaned dataset: 303 rows x 14 columns


In [4]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


In [6]:
df.dtypes.to_frame("dtype")

,dtype
age,float64
sex,float64
cp,float64
trestbps,float64
chol,float64
fbs,float64
restecg,float64
thalach,float64
exang,float64
oldpeak,float64


In [7]:
# Class balance: binary target derived from the UCI `num` column
# (num == 0 -> 0 "no disease", num > 0 -> 1 "disease indicated").
class_counts = df[TARGET].value_counts().sort_index()
class_pct = df[TARGET].value_counts(normalize=True).sort_index() * 100
balance_table = pd.DataFrame({
    "count": class_counts,
    "percent": class_pct.round(2),
})
balance_table.index.name = "target"
print(balance_table)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(class_counts.index.astype(str), class_counts.values,
              color=["#4c9f70", "#c44e52"], edgecolor="white")
ax.bar_label(bars)
ax.set_title("Heart Disease Class Balance")
ax.set_xlabel("Target (0 = not indicated, 1 = indicated)")
ax.set_ylabel("Records")
plt.tight_layout()
plt.show()

        count  percent
target                
0         164    54.13
1         139    45.87


/var/folders/y4/yl4j6_f952vc7tjvn8lslckc0000gn/T/ipykernel_34061/3344611258.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Missing values per column (raw '?' markers in the UCI file are parsed as NaN).
missing = df.isna().sum()
missing_table = missing[missing > 0].to_frame("missing_count")
missing_table["missing_percent"] = (missing_table["missing_count"] / len(df) * 100).round(3)
if missing_table.empty:
    print("No missing values in the cleaned dataset.")
missing_table

,missing_count,missing_percent
ca,4,1.32
thal,2,0.66


In [9]:
# Feature distributions
axes = df[FEATURES].hist(figsize=(14, 10), bins=18, edgecolor="white")
plt.suptitle("Heart Disease Features — Distributions", y=1.02)
plt.tight_layout()
plt.show()

/var/folders/y4/yl4j6_f952vc7tjvn8lslckc0000gn/T/ipykernel_34061/2732981741.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Correlation heatmap (features + target)
corr = df[FEATURES + [TARGET]].corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", square=True)
plt.title("Feature and Target Correlation Matrix")
plt.tight_layout()
plt.show()

/var/folders/y4/yl4j6_f952vc7tjvn8lslckc0000gn/T/ipykernel_34061/4066558956.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Observed EDA findings

Values below are the **actual observed numbers** from executing this notebook end-to-end
against `data/processed/heart_disease_cleaned.csv` (all code cells executed in place).

- Rows / columns: `303` rows / `13` features (plus `target`)
- Class balance: `164 no-disease (54.13%)` vs `139 disease (45.87%)`
- Missing values: `ca = 4 (1.32%)`, `thal = 2 (0.66%)` — all other columns complete
- Strongest target correlations: positive `thal +0.526`, `ca +0.460`, `exang +0.432` /
  negative `thalach −0.417` (`fbs +0.025`, `chol +0.085` near zero)
- Distribution notes: `chol` right-skewed with high outliers (median 241, max 564);
  `oldpeak` skewed (median 0.8, max 6.2); `trestbps` max 200